# ColLFM2 — Multimodal Retrieval with LFM2-VL

ColLFM2 uses LiquidAI's LFM2-VL (Mamba/SSM + attention hybrid) with
ColPali-style multi-vector pooling for text and image retrieval.

Multimodal late-interaction retrieval via LFM2-VL.

In [ ]:
import sys
import os
import time
import numpy as np
import torch

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, "plugins"))
sys.path.insert(0, os.path.join(ROOT, "models"))

## Load ColLFM2

In [ ]:
from vllm import LLM

llm = LLM(
    model="VAGOsolutions/SauerkrautLM-ColLFM2-450M-v0.1",
    runner="pooling",
    trust_remote_code=True,
    enforce_eager=True,
    gpu_memory_utilization=0.7,
    max_model_len=2048,
    skip_mm_profiling=True,
    mm_processor_cache_gb=1,
    limit_mm_per_prompt={"image": 1},
    enable_prefix_caching=False,
)

print("✅ ColLFM2 loaded")

## Encode Queries

In [ ]:
queries = [
    "What is the architecture of neural networks?",
    "How does retrieval augmented generation work?",
    "Explain the Mamba state-space model.",
    "What is late interaction in ColBERT?",
]

outputs = llm.encode(queries, pooling_task="token_embed")

for i, o in enumerate(outputs):
    emb = o.outputs.data
    if hasattr(emb, 'cpu'):
        emb = emb.cpu().numpy()
    else:
        emb = np.array(emb)
    if emb.ndim == 1:
        dim = 128
        emb = emb.reshape(-1, dim)
    print(f"  Query {i}: {emb.shape[0]} token vectors × {emb.shape[1]} dim")

print(f"✅ Encoded {len(queries)} queries")

## Throughput Benchmark

In [ ]:
text = "The quick brown fox jumps over the lazy dog. " * 20

for bs in [1, 8]:
    batch = [text] * bs
    for _ in range(5):
        llm.encode(batch, pooling_task="token_embed")
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(10):
        llm.encode(batch, pooling_task="token_embed")
    torch.cuda.synchronize()
    elapsed = (time.perf_counter() - t0) / 10
    print(f"Batch {bs:3d}: {bs/elapsed:.1f} req/s  ({elapsed*1000:.1f}ms)")

## Summary

ColLFM2 via vLLM Factory:
- Full multimodal ColPali pipeline via LFM2-VL
- LFM2-VL backbone (Mamba/SSM + attention hybrid)
- Multimodal: text + image via ColPali pooling
- cos_sim = 0.9999 (query), 0.9983 (image) parity